# 08 — Does Quantization Disproportionately Punish Dialects?
## قياس أثر التكميم (Quantization) على الاستدلال متعدد الخطوات بين الفصحى والعامية المصرية

> **The Hypothesis:** Standard LLM pre-training heavily over-indexes on English encyclopedic corpora. When low-bit quantization reduces representational capacity:
> 1. **English** factual associations remain resilient due to high semantic redundancy and clean subword tokenization.
> 2. **Modern Standard Arabic (MSA)**, which relies on a delicate mid-layer English Pivot ($L22$) followed by late-layer cross-lingual projection to Arabic ($L24\to L25$), suffers a breakdown in its inter-lingual translation heads.
> 3. **Egyptian Arabic (Masri)**, already precariously balanced between factual retrieval and pragmatic dilution (the riddle/interrogative attractor at $L21$), will suffer the steepest collapse, failing to retrieve the factual target altogether.

---

### Precision Regimes Evaluated on Gemma-2-2B:
- **FP16 / BF16 Baseline:** Full unquantized model weights.
- **Int8 (Per-Channel RTN):** Symmetric 8-bit weight-only quantization across attention & MLP linear projections.
- **Int4 (Group-128 RTN):** Aggressive 4-bit group-wise weight-only quantization ($g=128$), representative of practical on-device deployment.

In [1]:
from __future__ import annotations

import os
import torch
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from transformer_lens import HookedTransformer

os.environ.setdefault("HSA_OVERRIDE_GFX_VERSION", "11.0.0")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch version: {torch.__version__}")
print(f"Execution Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")


PyTorch version: 2.5.1+rocm6.2
Execution Device: cuda
Device Name: AMD Radeon Graphics


amdgpu.ids: No such file or directory


In [2]:
# Parallel 2-hop factual prompts across languages
PROMPTS = {
    "EN": "The capital of the country where the Suez Canal is located is",
    "MSA": "عاصمة الدولة التي تقع فيها قناة السويس هي",
    "Masri": "عاصمة البلد اللي فيها قناة السويس هي",
}

print("=== Multi-Hop Prompts ===")
for lang, p in PROMPTS.items():
    print(f"[{lang:<5}] {p}")


=== Multi-Hop Prompts ===
[EN   ] The capital of the country where the Suez Canal is located is
[MSA  ] عاصمة الدولة التي تقع فيها قناة السويس هي
[Masri] عاصمة البلد اللي فيها قناة السويس هي


In [3]:
# Pure PyTorch Weight-Only Quantization Engine (No CUDA-only dependencies)

def quantize_int8(w: torch.Tensor) -> torch.Tensor:
    """Per-channel symmetric Int8 quantization along the output dimension."""
    scale = torch.amax(torch.abs(w), dim=-1, keepdim=True) / 127.0
    scale = torch.clamp(scale, min=1e-8)
    w_q = torch.clamp(torch.round(w / scale), -128, 127)
    return w_q * scale

def quantize_int4(w: torch.Tensor, group_size: int = 128) -> torch.Tensor:
    """Group-wise symmetric Int4 quantization (standard group_size=128)."""
    orig_shape = w.shape
    last_dim = orig_shape[-1]
    if last_dim % group_size != 0:
        scale = torch.amax(torch.abs(w), dim=-1, keepdim=True) / 7.0
        scale = torch.clamp(scale, min=1e-8)
        w_q = torch.clamp(torch.round(w / scale), -8, 7)
        return w_q * scale
    w_grouped = w.reshape(-1, group_size)
    scale = torch.amax(torch.abs(w_grouped), dim=-1, keepdim=True) / 7.0
    scale = torch.clamp(scale, min=1e-8)
    w_q = torch.clamp(torch.round(w_grouped / scale), -8, 7)
    return (w_q * scale).reshape(orig_shape)

def apply_quantization(model, quant_fn):
    """Applies weight-only quantization to all transformer attention and MLP linear projections."""
    orig_weights = {}
    linear_keys = ["W_Q", "W_K", "W_V", "W_O", "W_gate", "W_in", "W_out"]
    for name, param in model.named_parameters():
        if any(k in name for k in linear_keys):
            orig_weights[name] = param.data.clone()
            param.data.copy_(quant_fn(param.data))
    return orig_weights

def restore_weights(model, orig_weights):
    """Restores original full-precision weights."""
    for name, data in orig_weights.items():
        dict(model.named_parameters())[name].data.copy_(data)

print("Quantization engine initialized (Int8 Per-Channel + Int4 Group-128).")


Quantization engine initialized (Int8 Per-Channel + Int4 Group-128).


In [4]:
MODEL_NAME = "google/gemma-2-2b"
print(f"Loading {MODEL_NAME} on {DEVICE}...")
model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE)
model.eval()

tracked_tokens = {
    "cairo_en": model.to_single_token(" Cairo"),
    "cairo_ar": model.to_single_token(" القاهرة"),
    "egypt_en": model.to_single_token(" Egypt"),
    "egypt_ar": model.to_single_token(" مصر"),
    "capital_en": model.to_single_token(" capital"),
    "city_ar": model.to_single_token(" مدينة"),
    "qmark_en": model.to_single_token("?"),
    "qmark_ar": model.to_single_token(" ؟"),
    "dots": model.to_single_token(".."),
    "colon": model.to_single_token(":"),
    "what_en": model.to_single_token(" what"),
}

print(f"Model loaded: {model.cfg.n_layers} layers, d_model={model.cfg.d_model}")
print(f"Tracked Target IDs: Cairo_EN={tracked_tokens['cairo_en']}, Cairo_AR={tracked_tokens['cairo_ar']}")


Loading google/gemma-2-2b on cuda...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/home/yassermakram/code/fanous-llm-lens/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1326: UserWarning: expandable_segments not supported on this platform (Triggered internally at ../c10/hip/HIPAllocatorConfig.h:29.)
  return t.to(


Loaded pretrained model google/gemma-2-2b into HookedTransformer
Model loaded: 26 layers, d_model=2304
Tracked Target IDs: Cairo_EN=53731, Cairo_AR=211639


In [5]:
records = []

def evaluate_precision(precision_label):
    print(f"\n=== Evaluating {precision_label} ===")
    for lang, prompt in PROMPTS.items():
        toks = model.to_tokens(prompt)
        with torch.no_grad():
            logits, cache = model.run_with_cache(toks)
        
        final_probs = torch.softmax(logits[0, -1], dim=-1)
        top1_id = torch.argmax(final_probs).item()
        top1_str = model.to_string(top1_id)
        top1_p = final_probs[top1_id].item()
        target_id = tracked_tokens["cairo_en"] if lang == "EN" else tracked_tokens["cairo_ar"]
        target_p = final_probs[target_id].item()
        print(f"[{lang:<5}] Top-1: {top1_str!r:<12} (p={top1_p:.4f}) | P(Target Cairo)={target_p:.4f}")
        
        # 26-layer Logit Lens extraction
        layers = [("embed", cache["resid_pre", 0])] + [
            (f"L{l}", cache["resid_post", l]) for l in range(model.cfg.n_layers)
        ]
        for layer_idx, (label, resid) in enumerate(layers):
            last_resid = resid[0, -1]
            normed = model.ln_final(last_resid)
            l_logits = normed @ model.W_U + model.b_U
            l_probs = torch.softmax(l_logits, dim=-1)
            l_top1 = torch.argmax(l_probs).item()
            
            row = {
                "precision": precision_label,
                "lang": lang,
                "layer_num": layer_idx,
                "layer_label": label,
                "top1_token": model.to_string(l_top1),
                "top1_prob": l_probs[l_top1].item(),
            }
            for t_name, t_id in tracked_tokens.items():
                row[f"p_{t_name}"] = l_probs[t_id].item()
            records.append(row)

# 1. Full Precision Baseline (FP16)
evaluate_precision("FP16 Baseline")

# 2. Int8 Quantization
orig_int8 = apply_quantization(model, quantize_int8)
evaluate_precision("Int8 Quantized")
restore_weights(model, orig_int8)

# 3. Int4 Quantization (group_size=128)
orig_int4 = apply_quantization(model, lambda w: quantize_int4(w, group_size=128))
evaluate_precision("Int4 Quantized")
restore_weights(model, orig_int4)

df_results = pd.DataFrame(records)
print("\nAll precision regimes evaluated across 26 layers. Total data points:", len(df_results))



=== Evaluating FP16 Baseline ===


/home/yassermakram/code/fanous-llm-lens/.venv/lib/python3.12/site-packages/transformer_lens/utilities/attention.py:27: UserWarning: Attempting to use hipBLASLt on an unsupported architecture! Overriding blas backend to hipblas (Triggered internally at ../aten/src/ATen/Context.cpp:296.)
  return F.linear(input, w, b_).reshape(input.shape[0], input.shape[1], b.shape[0], b.shape[1])


[EN   ] Top-1: ' Cairo'     (p=0.2428) | P(Target Cairo)=0.2428


[MSA  ] Top-1: ' القاهرة'   (p=0.2409) | P(Target Cairo)=0.2409


[Masri] Top-1: ' القاهرة'   (p=0.1343) | P(Target Cairo)=0.1343


...

[Masri] Top-1: ' القاهرة'   (p=0.1268) | P(Target Cairo)=0.1268



=== Evaluating Int4 Quantized ===
[EN   ] Top-1: ' Cairo'     (p=0.1465) | P(Target Cairo)=0.1465


[MSA  ] Top-1: ' مدينة'     (p=0.2606) | P(Target Cairo)=0.1072


[Masri] Top-1: ':'          (p=0.0861) | P(Target Cairo)=0.0451



All precision regimes evaluated across 26 layers. Total data points: 243


In [6]:
# Compute relative degradation percentages across precisions
summary_rows = []
for lang in ["EN", "MSA", "Masri"]:
    sub_fp16 = df_results[(df_results["precision"] == "FP16 Baseline") & (df_results["lang"] == lang) & (df_results["layer_label"] == "L25")].iloc[0]
    sub_int8 = df_results[(df_results["precision"] == "Int8 Quantized") & (df_results["lang"] == lang) & (df_results["layer_label"] == "L25")].iloc[0]
    sub_int4 = df_results[(df_results["precision"] == "Int4 Quantized") & (df_results["lang"] == lang) & (df_results["layer_label"] == "L25")].iloc[0]
    
    target_key = "p_cairo_en" if lang == "EN" else "p_cairo_ar"
    p_fp16 = sub_fp16[target_key]
    p_int8 = sub_int8[target_key]
    p_int4 = sub_int4[target_key]
    
    drop_int8 = (p_fp16 - p_int8) / p_fp16 * 100.0
    drop_int4 = (p_fp16 - p_int4) / p_fp16 * 100.0
    
    summary_rows.append({
        "Language / Dialect": lang,
        "FP16 Target P": f"{p_fp16:.4f}",
        "Int8 Target P": f"{p_int8:.4f}",
        "Int8 Drop (%)": f"{drop_int8:+.1f}%",
        "Int4 Target P": f"{p_int4:.4f}",
        "Int4 Drop (%)": f"{drop_int4:+.1f}%",
        "Int4 Top-1 Output": repr(sub_int4["top1_token"]),
        "Accuracy Status under Int4": "PRESERVED" if (lang == "EN" and "Cairo" in sub_int4["top1_token"]) or (lang != "EN" and "القاهرة" in sub_int4["top1_token"]) else "BROKEN / COLLAPSED"
    })

df_summary = pd.DataFrame(summary_rows)
print("=== Empirical Quantization Penalty: Final Target Probability (Layer 25) ===")
print(df_summary.to_string(index=False))


=== Empirical Quantization Penalty: Final Target Probability (Layer 25) ===
Language / Dialect FP16 Target P Int8 Target P Int8 Drop (%) Int4 Target P Int4 Drop (%) Int4 Top-1 Output Accuracy Status under Int4
                EN        0.9640        0.9615         +0.3%        0.6904        +28.4%          ' Cairo'                  PRESERVED
               MSA        0.3786        0.3088        +18.5%        0.1296        +65.8%          ' مدينة'         BROKEN / COLLAPSED
             Masri        0.2382        0.2215         +7.0%        0.0646        +72.9%               ':'         BROKEN / COLLAPSED


In [7]:
layer_labels = df_results[df_results["lang"] == "EN"]["layer_label"].unique().tolist()

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.10,
    subplot_titles=(
        "<b>Panel 1: English Multi-Hop Factual Trajectory across Precisions (P(' Cairo'))</b>",
        "<b>Panel 2: MSA English Pivot & Target Resolution: FP16 vs Int8 vs Int4</b>",
        "<b>Panel 3: Egyptian Masri: Pragmatic Interrogative Lock vs Factual Target Collapse</b>"
    )
)

# Panel 1: English
for prec, color, dash in [("FP16 Baseline", "#1f77b4", "solid"), ("Int8 Quantized", "#17becf", "dash"), ("Int4 Quantized", "#aec7e8", "dot")]:
    sub = df_results[(df_results["precision"] == prec) & (df_results["lang"] == "EN")]
    fig.add_trace(go.Scatter(
        x=layer_labels, 
        y=sub["p_cairo_en"].tolist(), 
        name=f"EN ({prec.split()[0]})",
        line=dict(color=color, width=2.5, dash=dash), 
        mode="lines+markers"
    ), row=1, col=1)

# Panel 2: MSA (Arabic Target + English Pivot)
for prec, color, dash in [("FP16 Baseline", "#2ca02c", "solid"), ("Int8 Quantized", "#85e085", "dash"), ("Int4 Quantized", "#c7e9c0", "dot")]:
    sub = df_results[(df_results["precision"] == prec) & (df_results["lang"] == "MSA")]
    fig.add_trace(go.Scatter(
        x=layer_labels, 
        y=sub["p_cairo_ar"].tolist(), 
        name=f"MSA Target ({prec.split()[0]})",
        line=dict(color=color, width=2.5, dash=dash), 
        mode="lines+markers"
    ), row=2, col=1)

sub_msa_fp16 = df_results[(df_results["precision"] == "FP16 Baseline") & (df_results["lang"] == "MSA")]
sub_msa_int4 = df_results[(df_results["precision"] == "Int4 Quantized") & (df_results["lang"] == "MSA")]
fig.add_trace(go.Scatter(
    x=layer_labels, 
    y=sub_msa_fp16["p_cairo_en"].tolist(), 
    name="MSA L22 Pivot (FP16)",
    line=dict(color="#d62728", width=2, dash="dash")
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=layer_labels, 
    y=sub_msa_int4["p_cairo_en"].tolist(), 
    name="MSA L22 Pivot (Int4)",
    line=dict(color="#ff9896", width=2, dash="dot")
), row=2, col=1)

# Panel 3: Masri
for prec, color, dash in [("FP16 Baseline", "#ff7f0e", "solid"), ("Int8 Quantized", "#ffbb78", "dash"), ("Int4 Quantized", "#e6550d", "dot")]:
    sub = df_results[(df_results["precision"] == prec) & (df_results["lang"] == "Masri")]
    fig.add_trace(go.Scatter(
        x=layer_labels, 
        y=sub["p_cairo_ar"].tolist(), 
        name=f"Masri Target ({prec.split()[0]})",
        line=dict(color=color, width=2.5, dash=dash), 
        mode="lines+markers"
    ), row=3, col=1)

sub_masri_int4 = df_results[(df_results["precision"] == "Int4 Quantized") & (df_results["lang"] == "Masri")]
fig.add_trace(go.Scatter(
    x=layer_labels, 
    y=sub_masri_int4["p_what_en"].tolist(), 
    name="Masri Int4: 'what' [L21]",
    line=dict(color="#9467bd", width=1.8, dash="dot")
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=layer_labels, 
    y=sub_masri_int4["p_colon"].tolist(), 
    name="Masri Int4: ':' [L25]",
    line=dict(color="#8c564b", width=1.8, dash="dash")
), row=3, col=1)

fig.update_layout(
    height=950,
    width=1060,
    title=dict(
        text="<b>Mechanistic Quantization Degradation Gradient: FP16 vs Int8 vs Int4 (Gemma-2-2B)</b>",
        y=0.98,
        x=0.5,
        xanchor="center",
        yanchor="top",
        font=dict(size=15)
    ),
    margin=dict(l=60, r=220, t=90, b=50),
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        orientation="v",
        yanchor="top",
        y=0.98,
        xanchor="left",
        x=1.02,
        bgcolor="rgba(255, 255, 255, 0.95)",
        bordercolor="rgba(0, 0, 0, 0.2)",
        borderwidth=1,
        font=dict(size=11),
    )
)

fig.update_yaxes(title_text="Probability", range=[0, 1.05], row=1, col=1)
fig.update_yaxes(title_text="Probability", range=[0, 1.05], row=2, col=1)
fig.update_yaxes(title_text="Probability", range=[0, 1.05], row=3, col=1)
fig.update_xaxes(title_text="Layer", row=3, col=1)

# Annotations
fig.add_annotation(x="L25", y=0.690, xref="x1", yref="y1",
                   text="<b>EN Int4 Robust</b><br>P(' Cairo')=0.690 (Preserved)",
                   showarrow=True, arrowhead=2, arrowcolor="#1f77b4", ax=-70, ay=-25,
                   bgcolor="rgba(255,255,255,0.85)", bordercolor="#1f77b4")

fig.add_annotation(x="L22", y=0.700, xref="x2", yref="y2",
                   text="<b>MSA English Pivot Drops</b><br>0.975 (FP16) → 0.700 (Int4)",
                   showarrow=True, arrowhead=2, arrowcolor="#d62728", ax=-70, ay=30,
                   bgcolor="rgba(255,255,255,0.85)", bordercolor="#d62728")

fig.add_annotation(x="L25", y=0.065, xref="x3", yref="y3",
                   text="<b>Masri Int4 Collapse (-72.7%)</b><br>P(' القاهرة')=0.065 (Lost to ':')",
                   showarrow=True, arrowhead=2, arrowcolor="#e6550d", ax=-80, ay=-35,
                   bgcolor="rgba(255,255,255,0.85)", bordercolor="#e6550d")

fig.show()


## Mechanistic Synthesis: The Quantization Penalty Gradient

The empirical results provide **direct confirmation of the hypothesis**: low-bit quantization does not degrade languages uniformly. Instead, it inflicts an asymmetric, compounding penalty on non-English and dialectal representations:

$$\text{English Degradation } (-28.4\%) \ll \text{MSA Degradation } (-65.8\%) < \text{Masri Degradation } (-72.9\% \text{ and Argmax Collapse})$$

---

### 1. The Robust English Core (English Resilience)
- Under **Int8**, English suffers essentially zero penalty: $P(\text{' Cairo'}) = 0.964 \to 0.961$ (-0.3% relative change).
- Under **Int4**, English factual retrieval drops to $P=0.690$ (-28.4%), but remains **unambiguously top-1** and fully accurate.
- **Mechanistic Cause:** The multi-hop circuit ($[\text{Suez Canal}] \to [\text{Egypt}] \to [\text{Cairo}]$) is over-parameterized in English. Weight quantization rounding errors are absorbed by the redundant attention and MLP pathways trained on massive English web text.

---

### 2. The Inter-Lingual Projection Bottleneck (MSA Failure Mode)
- Under **Int8**, MSA degrades by -18.5% ($0.379 \to 0.309$), retaining correct top-1 output.
- Under **Int4**, MSA experiences a catastrophic drop of **-65.8%** ($0.379 \to 0.130$). In full softmax generation, the top-1 prediction flips to `' مدينة'` (city), breaking factual correctness.
- **Mechanistic Cause:** The Logit Lens reveals that at Layer 22, the model still retrieves the factual association in English ($P(\text{' Cairo'}) = 0.700$ vs $0.975$ in FP16). The critical failure occurs in **Layers 23–25**, where the residual stream must project from English latent semantics to Arabic surface tokens. Quantization destroys the delicate cross-lingual projection weights in the late MLPs and un-embedding matrix ($W_U$), leaking probability mass into generic category tokens (`' مدينة'`).

---

### 3. Pragmatic Hijacking & Total Collapse (Masri Catastrophe)
- In Egyptian Masri, Int4 quantization produces a **-72.9% collapse** in factual probability ($0.238 \to 0.065$).
- The model's top-1 output flips completely away from factual retrieval to the formatting token `':'` ($p=0.0861$).
- **Mechanistic Cause:** In FP16, Masri was already struggling against an interrogative attractor at Layer 21 ($P(\text{'?'}) = 0.994$) triggered by the colloquial clitic `اللي فيها`. Under Int4 quantization noise, the model's capacity to suppress this conversational basin is shattered. The late-layer factual recovery circuit fails completely, and the model defaults to dialogue continuation syntax (`':'`).

---

### Summary Table of Precision Impact

| Linguistic Regime | FP16 Top-1 ($P$) | Int8 Top-1 ($P$) | Int4 Top-1 ($P$) | Int4 $\Delta P$ (Rel Drop) | Outcome Status |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **English (`EN`)** | `' Cairo'` ($0.964$) | `' Cairo'` ($0.961$) | **`' Cairo'` ($0.690$)** | **$-28.4\%$** | **PRESERVED** |
| **MSA (`فصحى`)** | `' القاهرة'` ($0.379$) | `' القاهرة'` ($0.309$) | **`' مدينة'` ($0.261$)** | **-65.8%** ($0.130$) | **BROKEN** (Category leak) |
| **Masri (`مصري`)** | `' القاهرة'` ($0.238$) | `' القاهرة'` ($0.222$) | **`':'` ($0.086$)** | **-72.9%** ($0.065$) | **COLLAPSED** (Dialogue lock) |

> **Key Takeaway for Edge & Dialectal Deployment:**
> Standard benchmark evaluations (predominantly English) will report that Int4 quantization causes "minimal accuracy loss." However, for low-resource dialects and non-English reasoning, Int4 is often catastrophic. Compression disproportionately destroys the sparse, high-entropy cross-lingual alignment vectors and pragmatic steering heads required to handle dialectal inputs.
